#### - IMPORTS

In [1]:
import os
import json
import time
import math
import datetime
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

#### - Global Config

In [2]:
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE      = 256
N               = 11                 # fractional bits
SCALE           = 2 ** N             # 2048
BASELINE_PTH    = "cnn_bns_hw_baseline.pth"
BASELINE_META   = "cnn_bns_hw_baseline_metadata.json"
EXPORT_DIR      = "quantized_params"

print(f"[Config] Device : {DEVICE}")
print(f"[Config] N={N}  Scale={SCALE}")
print()

[Config] Device : cpu
[Config] N=11  Scale=2048



#### - BASELINE MODEL ARCHITECTURE

In [3]:
class HWConvBlock(nn.Module):

  def __init__(
      self,
      in_channels:  int,
      out_channels: int,
      kernel_size:  int = 3,
      stride:       int = 1,
      padding:      int = 1,
      pool_size:    int = 2,
  ):
      super().__init__()
      self.conv = nn.Conv2d(
          in_channels, out_channels,
          kernel_size=kernel_size,
          stride=stride,
          padding=padding,
          bias=True,
      )

      self.relu = nn.ReLU(inplace=False)   # inplace=False preserves pre-relu tensor
      self.pool = nn.MaxPool2d(kernel_size=pool_size, stride=pool_size)
      nn.init.kaiming_normal_(self.conv.weight, mode='fan_out', nonlinearity='relu')
      nn.init.zeros_(self.conv.bias)

  def forward(
      self,
      x: torch.Tensor,
      return_raw: bool = False,
  ):

      # Step 1 — Convolution (pure MAC + bias)
      after_conv = self.conv(x)

      # Step 2 — ReLU
      after_relu = self.relu(after_conv)

      # Step 3 — MaxPool
      after_pool = self.pool(after_relu)

      if return_raw:
          raw = {
              "pre_relu"  : after_conv.detach().clone(),  # ← RNS must match this
              "pre_pool"  : after_relu.detach().clone(),
              "post_pool" : after_pool.detach().clone(),
          }
          return after_pool, raw

      return after_pool

class CNN_BNS_HW(nn.Module):
    def __init__(
        self,
        in_channels:  int,
        num_classes:  int,
        conv1_cfg:    tuple = (32, 3, 1, 1),
        conv2_cfg:    tuple = (64, 3, 1, 1),
        fc1_units:    int   = 256,
        dropout_rate: float = 0.5,
        input_hw:     tuple = (28, 28),
    ):
        super().__init__()

        c1_out, c1_k, c1_s, c1_p = conv1_cfg
        c2_out, c2_k, c2_s, c2_p = conv2_cfg

        self.conv_block1 = HWConvBlock(in_channels, c1_out, c1_k, c1_s, c1_p)
        self.conv_block2 = HWConvBlock(c1_out,      c2_out, c2_k, c2_s, c2_p)

        h_out    = input_hw[0] // 4
        w_out    = input_hw[1] // 4
        flat_dim = c2_out * h_out * w_out

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_dim, fc1_units),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate),
            nn.Linear(fc1_units, num_classes),
        )

        # Init FC layers
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

        total_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        self._arch_summary = {
            "type":         "CNN_BNS_HW (dual-conv)",
            "batch_norm":   False,
            "conv_bias":    True,
            "conv_block1":  f"{in_channels}→{c1_out} ch, k={c1_k}, pad={c1_p}",
            "conv_block2":  f"{c1_out}→{c2_out} ch, k={c2_k}, pad={c2_p}",
            "flatten_dim":  flat_dim,
            "fc1":          f"{flat_dim}→{fc1_units}",
            "fc2_out":      f"{fc1_units}→{num_classes}",
            "total_params": total_params,
        }

        print(f"\n[MODEL] CNN_BNS_HW — Dual ConvBlock (no BatchNorm, bias=True)")
        for k, v in self._arch_summary.items():
            print(f"  {k:<16}: {v}")
        print()

    def forward(
        self,
        x:               torch.Tensor,
        return_features: bool = False,
    ):
        fmaps = {}

        if return_features:
            fmaps["input"] = x.detach().clone()

        # ── Conv Block 1 ──────────────────────────────────────────────────
        if return_features:
            x, raw1 = self.conv_block1(x, return_raw=True)
            fmaps["conv1.pre_relu"]  = raw1["pre_relu"]
            fmaps["conv1.pre_pool"]  = raw1["pre_pool"]
            fmaps["conv1.post_pool"] = raw1["post_pool"]
        else:
            x = self.conv_block1(x)

        # ── Conv Block 2 ──────────────────────────────────────────────────
        if return_features:
            x, raw2 = self.conv_block2(x, return_raw=True)
            fmaps["conv2.pre_relu"]  = raw2["pre_relu"]
            fmaps["conv2.pre_pool"]  = raw2["pre_pool"]
            fmaps["conv2.post_pool"] = raw2["post_pool"]
        else:
            x = self.conv_block2(x)

        # ── Flatten capture ───────────────────────────────────────────────
        if return_features:
            fmaps["flatten"] = x.detach().clone().view(x.size(0), -1)

        # ── Classifier ────────────────────────────────────────────────────
        logits = self.classifier(x)

        if return_features:
            return logits, fmaps
        return logits

#### - PAPER-STYLE QUANTIZED CONV MODULE

In [4]:
class PaperStyleQuantConv2d(nn.Module):

    def __init__(self, conv_layer: nn.Conv2d, N: int = 11):
        super().__init__()

        self.N      = N
        self.scale  = 2 ** N
        self.stride  = conv_layer.stride
        self.padding = conv_layer.padding
        self.dilation = conv_layer.dilation
        self.groups  = conv_layer.groups

        with torch.no_grad():
            # Quantize weights:  q = ceil(w * 2^N)
            q_w = torch.ceil(conv_layer.weight.data * self.scale)
            self.register_buffer("q_weight", q_w)

            # Quantize bias if present
            if conv_layer.bias is not None:
                q_b = torch.ceil(conv_layer.bias.data * self.scale)
                self.register_buffer("q_bias", q_b)
            else:
                self.q_bias = None

    # ------------------------------------------------------------------
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Convolution in integer domain
        conv_int = F.conv2d(
            x,
            self.q_weight,
            bias   = self.q_bias,
            stride = self.stride,
            padding= self.padding,
            dilation=self.dilation,
            groups = self.groups,
        )
        # Rescale: floor(conv_int / 2^N) → approximately float output
        return torch.floor(conv_int / self.scale)

#### - QUANTIZED MODEL WRAPPER

In [5]:
class CNN_BNS_HW_Quantized(nn.Module):

    def __init__(self, base_model: CNN_BNS_HW,
                 quant_conv1: bool = False,
                 quant_conv2: bool = False,
                 N: int = 11):
        super().__init__()

        import copy

        # ── Block 1: copy block, optionally replace its .conv ────────
        self.conv_block1 = copy.deepcopy(base_model.conv_block1)
        if quant_conv1:
            # Patch only the conv layer inside the block; relu+pool unchanged
            self.conv_block1.conv = PaperStyleQuantConv2d(
                base_model.conv_block1.conv, N=N
            )

        # ── Block 2: same strategy ───────────────────────────────────
        self.conv_block2 = copy.deepcopy(base_model.conv_block2)
        if quant_conv2:
            self.conv_block2.conv = PaperStyleQuantConv2d(
                base_model.conv_block2.conv, N=N
            )

        # ── Classifier: shared FP32 weights ──────────────────────────
        self.classifier = base_model.classifier

    # ------------------------------------------------------------------
    def forward(self, x, return_features=False):
        feats = {}

        if return_features:
            feats["input"] = x.detach().clone()

        # ── Block 1 ──────────────────────────────────────────────────
        # Delegate entirely to HWConvBlock.forward(return_raw=True)
        # so we never manually re-implement conv/relu/pool sequencing.
        out1, raw1 = self.conv_block1(x, return_raw=True)
        if return_features:
            feats["conv1.pre_relu"]  = raw1["pre_relu"].detach().clone()
            feats["conv1.pre_pool"]  = raw1["pre_pool"].detach().clone()
            feats["conv1.post_pool"] = raw1["post_pool"].detach().clone()

        # ── Block 2 ──────────────────────────────────────────────────
        out2, raw2 = self.conv_block2(out1, return_raw=True)
        if return_features:
            feats["conv2.pre_relu"]  = raw2["pre_relu"].detach().clone()
            feats["conv2.pre_pool"]  = raw2["pre_pool"].detach().clone()
            feats["conv2.post_pool"] = raw2["post_pool"].detach().clone()

        # ── Classifier ───────────────────────────────────────────────
        flat = out2.flatten(1)
        if return_features:
            feats["flatten"] = flat.detach().clone()

        logits = self.classifier(out2)

        if return_features:
            return logits, feats
        return logits

#### - DATA LOADER

In [6]:
def get_test_loader():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    test_dataset = datasets.MNIST(
        root="./data", train=False, download=True, transform=transform
    )
    return DataLoader(test_dataset, batch_size=BATCH_SIZE,
                      shuffle=False, num_workers=2, pin_memory=True)

#### - EVALUATION UTILITIES

In [7]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total   = 0
    elapsed = 0.0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        t0 = time.perf_counter()
        logits = model(images)
        torch.cuda.synchronize() if device.type == "cuda" else None
        t1 = time.perf_counter()

        elapsed += (t1 - t0)
        preds    = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

    acc         = 100.0 * correct / total
    time_per_img = 1000.0 * elapsed / total   # ms
    return acc, time_per_img


#### - FEATURE-MAP ERROR ANALYSIS

In [8]:
@torch.no_grad()
def compute_feature_errors(fp32_model, quant_model, loader, device, num_batches=1):
    """
    Compares conv1.pre_relu and conv2.pre_relu between FP32 and a
    quantized variant over `num_batches` batches.

    Returns dict:
        {
          "conv1.pre_relu": {"mae": float, "mse": float, "max_abs": float},
          "conv2.pre_relu": {"mae": float, "mse": float, "max_abs": float},
        }
    """
    fp32_model.eval()
    quant_model.eval()

    errors = {
        "conv1.pre_relu": {"mae": [], "mse": [], "max_abs": []},
        "conv2.pre_relu": {"mae": [], "mse": [], "max_abs": []},
    }

    for batch_idx, (images, _) in enumerate(loader):
        if batch_idx >= num_batches:
            break

        images = images.to(device)

        _, feats_fp32  = fp32_model(images,  return_features=True)
        _, feats_quant = quant_model(images, return_features=True)

        for layer_key in ["conv1.pre_relu", "conv2.pre_relu"]:
            diff = (feats_fp32[layer_key] - feats_quant[layer_key]).abs()
            errors[layer_key]["mae"].append(diff.mean().item())
            errors[layer_key]["mse"].append((diff ** 2).mean().item())
            errors[layer_key]["max_abs"].append(diff.max().item())

    # Average over batches
    summary = {}
    for layer_key, vals in errors.items():
        summary[layer_key] = {
            "mae":     float(np.mean(vals["mae"])),
            "mse":     float(np.mean(vals["mse"])),
            "max_abs": float(np.mean(vals["max_abs"])),
        }
    return summary

#### - EXPORT QUANTIZED PARAMETERS

In [9]:
def export_quantized_params(base_model, accuracies, N):
    """
    Quantizes conv1 and conv2 weights/biases, saves as .npy integer arrays,
    and writes a JSON metadata file.
    """
    os.makedirs(EXPORT_DIR, exist_ok=True)
    scale = 2 ** N

    results = {}

    for layer_name, layer in [("conv1", base_model.conv_block1.conv),
                               ("conv2", base_model.conv_block2.conv)]:
        with torch.no_grad():
            w_int = torch.ceil(layer.weight.data * scale).cpu().numpy().astype(np.int32)
            np.save(os.path.join(EXPORT_DIR, f"{layer_name}_weight_int.npy"), w_int)

            if layer.bias is not None:
                b_int = torch.ceil(layer.bias.data * scale).cpu().numpy().astype(np.int32)
                np.save(os.path.join(EXPORT_DIR, f"{layer_name}_bias_int.npy"), b_int)
            else:
                b_int = None

        bits_needed = int(math.ceil(
            math.log2(max(abs(int(w_int.min())), abs(int(w_int.max()))) + 1) + 1
        ))

        results[layer_name] = {
            "shape":       list(w_int.shape),
            "w_min_int":   int(w_int.min()),
            "w_max_int":   int(w_int.max()),
            "bits_needed": bits_needed,
            "has_bias":    b_int is not None,
        }
        if b_int is not None:
            results[layer_name]["b_min_int"] = int(b_int.min())
            results[layer_name]["b_max_int"] = int(b_int.max())

    metadata = {
        "N":            N,
        "scale_factor": scale,
        "timestamp":    datetime.datetime.now().isoformat(),
        "layers":       results,
        "accuracies":   accuracies,
    }

    meta_path = os.path.join(EXPORT_DIR, "quantization_metadata.json")
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=4)

    print(f"\n[Export] Saved quantized params to '{EXPORT_DIR}/'")
    print(f"[Export] Metadata written to '{meta_path}'")
    return metadata

#### - PRETTY-PRINT TABLES

In [10]:
def print_accuracy_table(rows):
    """
    rows: list of dicts with keys:
        variant, conv1, conv2, accuracy, acc_loss, inf_time
    """
    hdr = (f"{'Variant':<22} {'Conv1':<8} {'Conv2':<8}"
           f" {'Acc (%)':>10} {'Acc Loss (%)':>13} {'Inf Time (ms/img)':>18}")
    sep = "─" * len(hdr)

    print("\n" + sep)
    print(" ACCURACY COMPARISON TABLE")
    print(sep)
    print(hdr)
    print(sep)
    for r in rows:
        print(f"{r['variant']:<22} {r['conv1']:<8} {r['conv2']:<8}"
              f" {r['accuracy']:>10.4f} {r['acc_loss']:>13.4f}"
              f" {r['inf_time']:>18.5f}")
    print(sep + "\n")


def print_feature_error_table(rows):
    """
    rows: list of dicts with keys:
        variant, layer, mae, mse, max_abs
    """
    hdr = (f"{'Variant':<22} {'Layer':<20}"
           f" {'MAE':>12} {'MSE':>14} {'Max Abs Err':>14}")
    sep = "─" * len(hdr)

    print(sep)
    print(" FEATURE-MAP ERROR TABLE  (conv1.pre_relu & conv2.pre_relu)")
    print(sep)
    print(hdr)
    print(sep)
    for r in rows:
        print(f"{r['variant']:<22} {r['layer']:<20}"
              f" {r['mae']:>12.6f} {r['mse']:>14.6f} {r['max_abs']:>14.6f}")
    print(sep + "\n")

#### - MAIN PIPELINE

In [16]:
def main():
    # ── 9.1  Load baseline weights ─────────────────────────────────────────
    print("[Step 1] Loading baseline model …")

    assert os.path.isfile(BASELINE_PTH),  f"Missing: {BASELINE_PTH}"
    assert os.path.isfile(BASELINE_META), f"Missing: {BASELINE_META}"

    with open(BASELINE_META) as f:
        meta = json.load(f)
    print(f"         Metadata: {meta}")

    fp32_base = fp32_base = CNN_BNS_HW(
    in_channels=1,
    num_classes=10,
    conv1_cfg=(32, 3, 1, 1),
    conv2_cfg=(64, 3, 1, 1),
    fc1_units=256,
    dropout_rate=0.5,
    input_hw=(28, 28),).to(DEVICE)

    state_dict = torch.load(BASELINE_PTH, map_location=DEVICE)
    fp32_base.load_state_dict(state_dict)
    fp32_base.eval()
    print("         Baseline loaded ✓\n")

    # ── 9.2  Data ──────────────────────────────────────────────────────────
    print("[Step 2] Building MNIST test loader …")
    test_loader = get_test_loader()
    print(f"         {len(test_loader.dataset)} test images\n")

    # ── 9.3  Build four model variants ────────────────────────────────────
    print("[Step 3] Building quantized model variants …")

    variants = [
        {
            "name":        "FP32 Baseline",
            "conv1_label": "FP32",
            "conv2_label": "FP32",
            "model":       fp32_base,
        },
        {
            "name":        "Q-Conv1 Only",
            "conv1_label": "Q-INT",
            "conv2_label": "FP32",
            "model": CNN_BNS_HW_Quantized(
                fp32_base, quant_conv1=True, quant_conv2=False, N=N
            ).to(DEVICE),
        },
        {
            "name":        "Q-Conv2 Only",
            "conv1_label": "FP32",
            "conv2_label": "Q-INT",
            "model": CNN_BNS_HW_Quantized(
                fp32_base, quant_conv1=False, quant_conv2=True, N=N
            ).to(DEVICE),
        },
        {
            "name":        "Q-Conv1 + Conv2",
            "conv1_label": "Q-INT",
            "conv2_label": "Q-INT",
            "model": CNN_BNS_HW_Quantized(
                fp32_base, quant_conv1=True, quant_conv2=True, N=N
            ).to(DEVICE),
        },
    ]

    for v in variants:
        v["model"].eval()
    print("         All variants ready ✓\n")

    # ── 9.4  Evaluate all variants ─────────────────────────────────────────
    print("[Step 4] Evaluating all variants on MNIST test set …")
    acc_rows = []
    accuracies = {}

    for v in variants:
        acc, t_img = evaluate(v["model"], test_loader, DEVICE)
        v["accuracy"]  = acc
        v["inf_time"]  = t_img
        accuracies[v["name"]] = acc
        print(f"         {v['name']:<22}: {acc:.4f}%  ({t_img:.5f} ms/img)")

    fp32_acc = variants[0]["accuracy"]
    for v in variants:
        v["acc_loss"] = fp32_acc - v["accuracy"]
        acc_rows.append({
            "variant":  v["name"],
            "conv1":    v["conv1_label"],
            "conv2":    v["conv2_label"],
            "accuracy": v["accuracy"],
            "acc_loss": v["acc_loss"],
            "inf_time": v["inf_time"],
        })

    print_accuracy_table(acc_rows)

    # ── 9.5  Feature-map error analysis ───────────────────────────────────
    print("[Step 5] Computing feature-map errors (first batch) …")
    feat_rows = []

    for v in variants[1:]:          # skip FP32 baseline
        errs = compute_feature_errors(
            fp32_base, v["model"], test_loader, DEVICE, num_batches=1
        )
        for layer_key, metrics in errs.items():
            feat_rows.append({
                "variant": v["name"],
                "layer":   layer_key,
                "mae":     metrics["mae"],
                "mse":     metrics["mse"],
                "max_abs": metrics["max_abs"],
            })

    print_feature_error_table(feat_rows)

    # ── 9.6  Export quantized parameters ──────────────────────────────────
    print("[Step 6] Exporting quantized integer parameters …")
    export_quantized_params(fp32_base, accuracies, N)

    # ============================================================
    # Export one real Conv1 MAC9 example for Verilog
    # ============================================================

    SCALE = 2 ** N

    test_loader = get_test_loader()
    images, labels = next(iter(test_loader))

    x_float = images[0:1]

    conv1 = fp32_base.conv_block1.conv

    w_float = conv1.weight.detach().cpu().numpy()
    b_float = conv1.bias.detach().cpu().numpy()

    # Integer quantization
    x_int = np.ceil(x_float.numpy() * SCALE).astype(np.int64)
    w_int = np.ceil(w_float * SCALE).astype(np.int64)

    # IMPORTANT:
    # bias uses SCALE^2 for integer MAC
    b_hw_int = np.ceil(b_float * SCALE * SCALE).astype(np.int64)

    # Padding because Conv1 uses padding=1
    x_pad = np.pad(
        x_int,
        ((0,0),(0,0),(1,1),(1,1)),
        mode='constant'
    )

    # Extract top-left patch
    patch = x_pad[0,0,0:3,0:3]

    # Filter 0
    kernel = w_int[0,0]

    bias = int(b_hw_int[0])

    golden = int((patch * kernel).sum() + bias)

    print("\nPATCH:")
    print(patch)

    print("\nKERNEL:")
    print(kernel)

    print("\nBIAS:")
    print(bias)

    print("\nGOLDEN:")
    print(golden)
    # ============================================================
    # Export full Conv1 layer: all 32 filters, all 28x28 positions
    # Total tests = 32 * 784 = 25088
    # ============================================================

    SCALE = 2 ** N

    test_loader = get_test_loader()
    images, labels = next(iter(test_loader))

    x_float = images[0:1]

    conv1 = fp32_base.conv_block1.conv
    w_float = conv1.weight.detach().cpu().numpy()
    b_float = conv1.bias.detach().cpu().numpy()

    x_int = np.ceil(x_float.numpy() * SCALE).astype(np.int64)
    w_int = np.ceil(w_float * SCALE).astype(np.int64)
    b_hw_int = np.ceil(b_float * SCALE * SCALE).astype(np.int64)

    x_pad = np.pad(
        x_int,
        ((0,0), (0,0), (1,1), (1,1)),
        mode="constant"
    )

    patches = []
    goldens = []

    for filter_idx in range(32):
        kernel = w_int[filter_idx, 0, :, :]
        bias = int(b_hw_int[filter_idx])

        for row in range(28):
            for col in range(28):
                patch = x_pad[0, 0, row:row+3, col:col+3]
                golden = int((patch * kernel).sum() + bias)

                values = list(patch.flatten()) + list(kernel.flatten()) + [bias]
                patches.append(values)
                goldens.append(golden)

    patches = np.array(patches, dtype=np.int64)
    goldens = np.array(goldens, dtype=np.int64)

    def to_hex32(v):
        return format(np.uint32(v).item(), "08x")

    with open("conv1_all_filters_mac9_inputs_hex.txt", "w") as f:
        for row in patches:
            f.write(" ".join(to_hex32(v) for v in row) + "\n")

    with open("conv1_all_filters_mac9_goldens_hex.txt", "w") as f:
        for v in goldens:
            f.write(to_hex32(v) + "\n")

    print("Saved:")
    print("conv1_all_filters_mac9_inputs_hex.txt")
    print("conv1_all_filters_mac9_goldens_hex.txt")
    print("Number of tests:", len(goldens))
    print("Golden min:", goldens.min())
    print("Golden max:", goldens.max())

    # ============================================================
    # Export full Conv1 feature map test data for Verilog
    # Filter 0, all 28x28 output positions
    # ============================================================

    SCALE = 2 ** N

    test_loader = get_test_loader()
    images, labels = next(iter(test_loader))

    x_float = images[0:1]  # first MNIST image

    conv1 = fp32_base.conv_block1.conv
    w_float = conv1.weight.detach().cpu().numpy()
    b_float = conv1.bias.detach().cpu().numpy()

    x_int = np.ceil(x_float.numpy() * SCALE).astype(np.int64)
    w_int = np.ceil(w_float * SCALE).astype(np.int64)
    b_hw_int = np.ceil(b_float * SCALE * SCALE).astype(np.int64)

    filter_idx = 0

    x_pad = np.pad(
        x_int,
        ((0,0), (0,0), (1,1), (1,1)),
        mode="constant"
    )

    kernel = w_int[filter_idx, 0, :, :]
    bias = int(b_hw_int[filter_idx])

    patches = []
    goldens = []

    for row in range(28):
        for col in range(28):
            patch = x_pad[0, 0, row:row+3, col:col+3]
            golden = int((patch * kernel).sum() + bias)

            values = list(patch.flatten()) + list(kernel.flatten()) + [bias]
            patches.append(values)
            goldens.append(golden)

    patches = np.array(patches, dtype=np.int64)
    goldens = np.array(goldens, dtype=np.int64)

    np.savetxt("conv1_filter0_mac9_inputs.txt", patches, fmt="%d")
    np.savetxt("conv1_filter0_mac9_goldens.txt", goldens, fmt="%d")

    print("Saved:")
    print("conv1_filter0_mac9_inputs.txt")
    print("conv1_filter0_mac9_goldens.txt")
    print("Number of tests:", len(goldens))
    print("Golden min:", goldens.min())
    print("Golden max:", goldens.max())


    # ============================================================
    # Export full Conv1 layer: all 32 filters, all 28x28 positions
    # Total tests = 32 * 784 = 25088
    # ============================================================

    SCALE = 2 ** N

    test_loader = get_test_loader()
    images, labels = next(iter(test_loader))

    x_float = images[0:1]

    conv1 = fp32_base.conv_block1.conv
    w_float = conv1.weight.detach().cpu().numpy()
    b_float = conv1.bias.detach().cpu().numpy()

    x_int = np.ceil(x_float.numpy() * SCALE).astype(np.int64)
    w_int = np.ceil(w_float * SCALE).astype(np.int64)
    b_hw_int = np.ceil(b_float * SCALE * SCALE).astype(np.int64)

    x_pad = np.pad(
        x_int,
        ((0,0), (0,0), (1,1), (1,1)),
        mode="constant"
    )

    patches = []
    goldens = []

    for filter_idx in range(32):
        kernel = w_int[filter_idx, 0, :, :]
        bias = int(b_hw_int[filter_idx])

        for row in range(28):
            for col in range(28):
                patch = x_pad[0, 0, row:row+3, col:col+3]
                golden = int((patch * kernel).sum() + bias)

                values = list(patch.flatten()) + list(kernel.flatten()) + [bias]
                patches.append(values)
                goldens.append(golden)

    patches = np.array(patches, dtype=np.int64)
    goldens = np.array(goldens, dtype=np.int64)

    def to_hex32(v):
        return format(np.uint32(v).item(), "08x")

    with open("conv1_all_filters_mac9_inputs_hex.txt", "w") as f:
        for row in patches:
            f.write(" ".join(to_hex32(v) for v in row) + "\n")

    with open("conv1_all_filters_mac9_goldens_hex.txt", "w") as f:
        for v in goldens:
            f.write(to_hex32(v) + "\n")

    print("Saved:")
    print("conv1_all_filters_mac9_inputs_hex.txt")
    print("conv1_all_filters_mac9_goldens_hex.txt")
    print("Number of tests:", len(goldens))
    print("Golden min:", goldens.min())
    print("Golden max:", goldens.max())

    # ============================================================
    # Export ONE Conv2 output pixel test for Verilog
    # Conv2: 32 input channels, 3x3 kernel = 288 MACs
    # ============================================================

    SCALE = 2 ** N

    test_loader = get_test_loader()
    images, labels = next(iter(test_loader))

    x_float = images[0:1]  # first MNIST image

    conv1 = fp32_base.conv_block1.conv
    conv2 = fp32_base.conv_block2.conv

    # -------------------------
    # Quantize weights
    # -------------------------
    w1_int = np.ceil(conv1.weight.detach().cpu().numpy() * SCALE).astype(np.int64)
    b1_hw  = np.ceil(conv1.bias.detach().cpu().numpy() * SCALE * SCALE).astype(np.int64)

    w2_int = np.ceil(conv2.weight.detach().cpu().numpy() * SCALE).astype(np.int64)
    b2_hw  = np.ceil(conv2.bias.detach().cpu().numpy() * SCALE * SCALE).astype(np.int64)

    # -------------------------
    # Quantize input
    # -------------------------
    x_int = np.ceil(x_float.numpy() * SCALE).astype(np.int64)

    # ============================================================
    # Integer Conv1 manually
    # Output raw scale = SCALE^2
    # Then divide by SCALE to return to SCALE domain for Conv2 input
    # ============================================================

    x_pad = np.pad(
        x_int,
        ((0,0), (0,0), (1,1), (1,1)),
        mode="constant"
    )

    conv1_raw = np.zeros((32, 28, 28), dtype=np.int64)

    for f in range(32):
        kernel = w1_int[f, 0, :, :]
        bias = int(b1_hw[f])

        for r in range(28):
            for c in range(28):
                patch = x_pad[0, 0, r:r+3, c:c+3]
                conv1_raw[f, r, c] = int((patch * kernel).sum() + bias)

    # Fixed-point rescale
    conv1_scaled = np.floor_divide(conv1_raw, SCALE)

    # ReLU
    conv1_relu = np.maximum(conv1_scaled, 0)

    # MaxPool 2x2 stride 2
    conv1_pool = np.zeros((32, 14, 14), dtype=np.int64)

    for ch in range(32):
        for r in range(14):
            for c in range(14):
                block = conv1_relu[ch, r*2:r*2+2, c*2:c*2+2]
                conv1_pool[ch, r, c] = block.max()

    # ============================================================
    # Conv2 one output pixel
    # Choose output filter 0, row 0, col 0
    # ============================================================

    filter_idx = 0
    row = 0
    col = 0

    conv1_pool_pad = np.pad(
        conv1_pool,
        ((0,0), (1,1), (1,1)),
        mode="constant"
    )

    x_values = []
    w_values = []

    for ch in range(32):
        patch = conv1_pool_pad[ch, row:row+3, col:col+3]
        kernel = w2_int[filter_idx, ch, :, :]

        x_values.extend(list(patch.flatten()))
        w_values.extend(list(kernel.flatten()))

    bias = int(b2_hw[filter_idx])

    x_values = np.array(x_values, dtype=np.int64)
    w_values = np.array(w_values, dtype=np.int64)

    golden = int((x_values * w_values).sum() + bias)

    print("\n[CONV2 ONE-PIXEL EXPORT]")
    print("Label:", labels[0].item())
    print("Conv2 filter:", filter_idx)
    print("Output position:", row, col)
    print("Number of MAC terms:", len(x_values))
    print("Bias:", bias)
    print("Golden:", golden)
    print("Min x:", x_values.min(), "Max x:", x_values.max())
    print("Min w:", w_values.min(), "Max w:", w_values.max())

    print("\nDynamic range check:")
    print("Golden abs:", abs(golden))
    print("RNS signed limit:", 1072873614)

    if abs(golden) < 1072873614:
        print("SAFE: inside RNS signed range")
    else:
        print("WARNING: outside RNS signed range")

    def to_hex32(v):
        return format(int(v) & 0xFFFFFFFF, "08x")

    with open("conv2_one_pixel_x_hex.txt", "w") as f:
        for v in x_values:
            f.write(to_hex32(v) + "\n")

    with open("conv2_one_pixel_w_hex.txt", "w") as f:
        for v in w_values:
            f.write(to_hex32(v) + "\n")

    with open("conv2_one_pixel_bias_hex.txt", "w") as f:
        f.write(to_hex32(bias) + "\n")

    with open("conv2_one_pixel_golden_hex.txt", "w") as f:
        f.write(to_hex32(golden) + "\n")

    # ============================================================
    # Export Conv2 filter 0 full feature map
    # 14x14 = 196 outputs, each output has 288 MAC terms
    # ============================================================

    filter_idx = 0

    conv2_inputs = []
    conv2_weights = []
    conv2_biases = []
    conv2_goldens = []

    for row in range(14):
        for col in range(14):

            x_values = []
            w_values = []

            for ch in range(32):
                patch = conv1_pool_pad[ch, row:row+3, col:col+3]
                kernel = w2_int[filter_idx, ch, :, :]

                x_values.extend(list(patch.flatten()))
                w_values.extend(list(kernel.flatten()))

            x_values = np.array(x_values, dtype=np.int64)
            w_values = np.array(w_values, dtype=np.int64)

            bias = int(b2_hw[filter_idx])
            golden = int((x_values * w_values).sum() + bias)

            conv2_inputs.append(x_values)
            conv2_weights.append(w_values)
            conv2_biases.append(bias)
            conv2_goldens.append(golden)

    conv2_inputs = np.array(conv2_inputs, dtype=np.int64)
    conv2_weights = np.array(conv2_weights, dtype=np.int64)
    conv2_biases = np.array(conv2_biases, dtype=np.int64)
    conv2_goldens = np.array(conv2_goldens, dtype=np.int64)

    def to_hex32(v):
        return format(int(v) & 0xFFFFFFFF, "08x")

    with open("conv2_filter0_x_hex.txt", "w") as f:
        for row in conv2_inputs:
            for v in row:
                f.write(to_hex32(v) + "\n")

    with open("conv2_filter0_w_hex.txt", "w") as f:
        for row in conv2_weights:
            for v in row:
                f.write(to_hex32(v) + "\n")

    with open("conv2_filter0_bias_hex.txt", "w") as f:
        for v in conv2_biases:
            f.write(to_hex32(v) + "\n")

    with open("conv2_filter0_goldens_hex.txt", "w") as f:
        for v in conv2_goldens:
            f.write(to_hex32(v) + "\n")

    print("Saved Conv2 filter0 full map files:")
    print("conv2_filter0_x_hex.txt")
    print("conv2_filter0_w_hex.txt")
    print("conv2_filter0_bias_hex.txt")
    print("conv2_filter0_goldens_hex.txt")
    print("Number of outputs:", len(conv2_goldens))
    print("Golden min:", conv2_goldens.min())
    print("Golden max:", conv2_goldens.max())


    # ── 9.7  Final summary ─────────────────────────────────────────────────
    print("=" * 60)
    print("  QUANTIZATION SUMMARY")
    print("=" * 60)
    print(f"  Fractional bits  N  = {N}")
    print(f"  Scale factor       = {SCALE}")
    print(f"  FP32 accuracy      = {fp32_acc:.4f}%")
    best_q = min(variants[1:], key=lambda v: v["acc_loss"])
    worst_q = max(variants[1:], key=lambda v: v["acc_loss"])
    print(f"  Best  quantized    = {best_q['name']}  (loss {best_q['acc_loss']:.4f}%)")
    print(f"  Worst quantized    = {worst_q['name']} (loss {worst_q['acc_loss']:.4f}%)")
    print("=" * 60)

    # ============================================================
    # Export FULL Conv2 layer
    # 64 filters × 14×14 outputs
    # ============================================================

    NUM_FILTERS = 64
    OUT_HW = 14

    conv2_inputs = []
    conv2_weights = []
    conv2_biases = []
    conv2_goldens = []

    for filter_idx in range(NUM_FILTERS):

        for row in range(OUT_HW):

            for col in range(OUT_HW):

                x_values = []
                w_values = []

                for ch in range(32):

                    patch = conv1_pool_pad[ch, row:row+3, col:col+3]
                    kernel = w2_int[filter_idx, ch, :, :]

                    x_values.extend(list(patch.flatten()))
                    w_values.extend(list(kernel.flatten()))

                x_values = np.array(x_values, dtype=np.int64)
                w_values = np.array(w_values, dtype=np.int64)

                bias = int(b2_hw[filter_idx])

                golden = int((x_values * w_values).sum() + bias)

                conv2_inputs.append(x_values)
                conv2_weights.append(w_values)
                conv2_biases.append(bias)
                conv2_goldens.append(golden)

    conv2_inputs = np.array(conv2_inputs, dtype=np.int64)
    conv2_weights = np.array(conv2_weights, dtype=np.int64)
    conv2_biases = np.array(conv2_biases, dtype=np.int64)
    conv2_goldens = np.array(conv2_goldens, dtype=np.int64)

    print("Total outputs:", len(conv2_goldens))

    def to_hex32(v):
        return format(int(v) & 0xFFFFFFFF, "08x")

    with open("conv2_all_filters_x_hex.txt", "w") as f:
        for row in conv2_inputs:
            for v in row:
                f.write(to_hex32(v) + "\n")

    with open("conv2_all_filters_w_hex.txt", "w") as f:
        for row in conv2_weights:
            for v in row:
                f.write(to_hex32(v) + "\n")

    with open("conv2_all_filters_bias_hex.txt", "w") as f:
        for v in conv2_biases:
            f.write(to_hex32(v) + "\n")

    with open("conv2_all_filters_goldens_hex.txt", "w") as f:
        for v in conv2_goldens:
            f.write(to_hex32(v) + "\n")

    print("Saved FULL Conv2 layer files.")


# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    main()

[Step 1] Loading baseline model …
         Metadata: {'run_id': '20260429_050342', 'model_variant': 'dual', 'config': {'seed': '42', 'dataset': 'MNIST', 'data_dir': './data', 'num_workers': '2', 'batch_size': '64', 'learning_rate': '0.001', 'num_epochs': '10', 'weight_decay': '0.0001', 'conv1': '(32, 3, 1, 1)', 'conv2': '(64, 3, 1, 1)', 'fc1_units': '256', 'dropout_rate': '0.5', 'model_variant': 'dual', 'model_save_path': './cnn_bns_hw_baseline.pth', 'metadata_save_path': './cnn_bns_hw_baseline_metadata.json', 'device': 'cpu'}, 'architecture': {'type': 'CNN_BNS_HW (dual-conv)', 'batch_norm': False, 'conv_bias': True, 'conv_block1': '1→32 ch, k=3, pad=1', 'conv_block2': '32→64 ch, k=3, pad=1', 'flatten_dim': 3136, 'fc1': '3136→256', 'fc2_out': '256→10', 'total_params': 824458}, 'total_params': 824458, 'training_history': [{'epoch': 1, 'train_loss': 0.166468, 'train_acc': 94.8033, 'test_acc': 98.76, 'lr': 0.001}, {'epoch': 2, 'train_loss': 0.05986, 'train_acc': 98.1633, 'test_acc': 99.11